# Project R.E.M. — Analyse
## Regulation of Emotion through Music

Dit notebook bundelt alle analyseresultaten van Project R.E.M. in één overzicht.  

**Wat wordt onderzocht?**  
Kunnen gepersonaliseerde muziekplaylists op basis van het ISO-principe fysiologische stress verminderen en de stemming verbeteren?

**Structuur:**

| # | Sectie | Inhoud |
|---|--------|--------|
| 1 | Setup & stijl | Imports, paden, kleurpalet |
| 2 | Deelnemersoverzicht | Databeschikbaarheid per deelnemer |
| 3 | Circadiane baselines | Persoonlijke 24u stresspatronen |
| 4 | Ruwe sessietraces | Stress voor/tijdens/na muziekluisteren |
| 5 | Stemmingsdistributie | Stemmingsverandering per afspeellijsttype |
| 6 | Lang-termijntrend | Stressdeviation over de tijd |
| 7 | Herstelanalyse | Tau-voordeel: herstelt stress sneller met muziek? |
| 8 | ML-modelresultaten | Voorspelling stemming en stress |
| 9 | Significantietoetsen | Zijn effecten statistisch aantoonbaar? |
| 10 | Bayesiaanse aanbeveling | Welk afspeellijsttype past bij wie? |

---
*Data: Garmin/Huawei smartwatch + Spotify Exportify + Google Forms check-ins*  
*Deelnemers zijn geanonimiseerd met fruitcodenamen*

---
## Sectie 1 — Setup & stijl

Imports, kleurpalet, paden en databeschikbaarheidscheck.  
Dit notebook laadt uitsluitend vooraf berekende CSV-bestanden — er worden geen analysemodellen opnieuw uitgevoerd.

In [3]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import arviz as az

warnings.filterwarnings('ignore', message='Glyph .* missing from font')

# ── Paden ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path().resolve().parent
DATA_ROOT     = PROJECT_ROOT / 'data'
ANALYSIS_ROOT = DATA_ROOT / 'analysis'
COMBINED_DIR  = ANALYSIS_ROOT / 'circadian_baselines'
BAYESIAN_DIR  = ANALYSIS_ROOT / 'bayesian_recommender'
WEARABLES_DIR = DATA_ROOT / 'wearables'
CHECKIN_PATH  = DATA_ROOT / 'checkins' / 'Check-in_formulier_REM.csv'

# ── Okabe-Ito kleurenpalet ────────────────────────────────────────────────────
# Ontworpen voor kleurenblindheid (deuteranopie, protanopie, tritanopie).
# Bron: Okabe & Ito (2008) https://jfly.uni-koeln.de/color/
OI = {
    'black':          '#000000',
    'orange':         '#E69F00',
    'sky_blue':       '#56B4E9',
    'bluish_green':   '#009E73',
    'yellow':         '#F0E442',
    'blue':           '#0072B2',
    'vermilion':      '#D55E00',
    'reddish_purple': '#CC79A7',
}

PARTICIPANTS = {
    'bosbes':      {'color': OI['sky_blue'],       'emoji': '🫐'},
    'kokosnoot':   {'color': OI['orange'],          'emoji': '🥥'},
    'limoen':      {'color': OI['bluish_green'],    'emoji': '🍋'},
    'peer':        {'color': OI['reddish_purple'],  'emoji': '🍐'},
    'kiwi':        {'color': OI['vermilion'],       'emoji': '🥝'},
    'watermeloen': {'color': OI['blue'],            'emoji': '🍉'},
}

PLAYLIST_COLORS = {
    'Calm':    OI['sky_blue'],
    'Energy':  OI['vermilion'],
    'Neutral': OI['yellow'],
}

STATE_COLORS = {
    'Sleep':  OI['blue'],
    'Rest':   OI['sky_blue'],
    'Light':  OI['bluish_green'],
    'Medium': OI['orange'],
    'Heavy':  OI['vermilion'],
}

# ── Donker thema (identiek aan recovery_analysis.ipynb) ──────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1218', 'axes.facecolor': '#181e2a',
    'axes.edgecolor': '#232b3a',   'axes.labelcolor': '#c9d1d9',
    'axes.grid': True,             'grid.color': '#232b3a',
    'grid.linewidth': 0.5,         'text.color': '#c9d1d9',
    'xtick.color': '#586475',      'ytick.color': '#586475',
    'legend.facecolor': '#181e2a', 'legend.edgecolor': '#232b3a',
    'font.family': 'monospace',    'font.size': 9,
    'figure.dpi': 120,
})

print("✓ Setup voltooid")

✓ Setup voltooid


In [4]:
# ── Databeschikbaarheidscheck ─────────────────────────────────────────────────
# Per deelnemer: welke bestanden bestaan, hoeveel sessies, welk apparaat.

GARMIN_PARTICIPANTS  = ['bosbes', 'kokosnoot', 'peer']
HUAWEI_PARTICIPANTS  = ['limoen']
CHECKIN_PARTICIPANTS = ['bosbes', 'kiwi', 'kokosnoot', 'limoen', 'peer', 'watermeloen']

checks = {
    'feature_matrix':    COMBINED_DIR / 'feature_matrix.csv',
    'significance':      COMBINED_DIR / 'significance_tests.csv',
    'bayesian_trace':    BAYESIAN_DIR / 'trace.nc',
    'recommendations':   BAYESIAN_DIR / 'recommendations.json',
}

# Laad feature matrix voor sessietelling
fm = pd.read_csv(checks['feature_matrix']) if checks['feature_matrix'].exists() else pd.DataFrame()

print("=" * 62)
print(f"{'Deelnemer':<14} {'Apparaat':<10} {'Sessies':<9} {'Biometrie':<11} {'Herstel':<9} {'Check-ins'}")
print("-" * 62)

for p in sorted(PARTICIPANTS.keys()):
    emoji = PARTICIPANTS[p]['emoji']

    # Apparaat
    if p in GARMIN_PARTICIPANTS:
        device = 'Garmin'
    elif p in HUAWEI_PARTICIPANTS:
        device = 'Huawei'
    else:
        device = '—'

    # Sessies in feature matrix
    n_sessions = len(fm[fm['participant'] == p]) if not fm.empty and p in fm['participant'].values else 0

    # Biometrische data
    bio_path = WEARABLES_DIR / p / 'processed' / ('garmin_minute_stress.csv' if p in GARMIN_PARTICIPANTS else 'huawei_minute_stress.csv')
    bio_ok = '✓' if bio_path.exists() else '✗'

    # Hersteldata
    rec_path = ANALYSIS_ROOT / p / 'session_effects.csv'
    rec_ok = '✓' if rec_path.exists() else '✗'

    # Check-ins
    checkin_ok = '✓' if p in CHECKIN_PARTICIPANTS else '✗'

    print(f"{emoji} {p:<12} {device:<10} {n_sessions:<9} {bio_ok:<11} {rec_ok:<9} {checkin_ok}")

print("=" * 62)
print()
print("Noten:")
print("  • limoen: Huawei — herstelanalyse (pipeline.py) niet ondersteund → sectie 7 uitgesloten")
print("  • kiwi, watermeloen: alleen check-in data, geen biometrie → meeste secties uitgesloten")
print("  • bosbes: slechts 4 sessies — resultaten indicatief, niet conclusief")

Deelnemer      Apparaat   Sessies   Biometrie   Herstel   Check-ins
--------------------------------------------------------------
🫐 bosbes       Garmin     4         ✓           ✓         ✓
🥝 kiwi         —          0         ✗           ✗         ✓
🥥 kokosnoot    Garmin     40        ✓           ✓         ✓
🍋 limoen       Huawei     8         ✓           ✗         ✓
🍐 peer         Garmin     30        ✓           ✓         ✓
🍉 watermeloen  —          0         ✗           ✗         ✓

Noten:
  • limoen: Huawei — herstelanalyse (pipeline.py) niet ondersteund → sectie 7 uitgesloten
  • kiwi, watermeloen: alleen check-in data, geen biometrie → meeste secties uitgesloten
  • bosbes: slechts 4 sessies — resultaten indicatief, niet conclusief
